# OSM vs ENTSO-E GridKit

## Part I: Setup

### Libraries

In [ ]:
# loading required libraries
import pypsa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cartopy import crs as ccrs
import xarray as xr
import folium
from folium import plugins
import geopandas as gpd
from branca.element import Figure
import branca.colormap as cm
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from shapely import box, normalize
from shapely.geometry import Polygon, LineString, Point, MultiPoint
from shapely.wkt import loads
from tqdm.auto import tqdm
import os


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

### PyPSA Network

In [ ]:
network_entsoe = pypsa.Network("networks/entsoe_europe/base.nc")
network_osm = pypsa.Network("networks/osm_europe/base.nc")

print(network_entsoe)
print(network_osm)

In [ ]:
# Quick plots of the networks
fig, axes = plt.subplots(1, 2, figsize = (10, 5), subplot_kw = {"projection": ccrs.EqualEarth()})

network_entsoe.plot(ax = axes[0], title = "ENTSO-E\n", bus_sizes = 0, line_widths = 0.3, link_widths = 0.3)
network_osm.plot(ax = axes[1], title = "OSM\n", bus_sizes = 0, line_widths = 0.3, link_widths = 0.3)

# Show the plot
plt.show()

In [ ]:
# Define function to create lines using the start bus and end bus coordinates.
# This method yields point-to-point straight lines, returned as list of 
# shapely.geometry.linestring.LineString

def lineGeometry(network):
    geometry = [LineString(
        [(network.buses.loc[row.bus0]["x"], network.buses.loc[row.bus0]["y"]), 
         (network.buses.loc[row.bus1]["x"], network.buses.loc[row.bus1]["y"])]
         ) \
            for idx, row in network.lines.iterrows()]
    return geometry

def linkGeometry(network):
    geometry = [LineString(
        [(network.buses.loc[row.bus0]["x"], network.buses.loc[row.bus0]["y"]), 
         (network.buses.loc[row.bus1]["x"], network.buses.loc[row.bus1]["y"])]
         ) \
            for idx, row in network.links.iterrows()]
    return geometry

In [ ]:
# Use shapely.wkt.loads to convert the string representation of the geometry to
# a shapely object. This is only available for OSM data from pypsa-earth.
network_osm.lines["geometry"] = network_osm.lines["geometry"].apply(loads)

In [ ]:
# Create a GeoDataFrame for the ENTSO-E network using the lineGeometry function
gdf_entsoe_lines = gpd.GeoDataFrame(network_entsoe.lines, geometry = lineGeometry(network_entsoe), crs = "EPSG:4326")
gdf_entsoe_links = gpd.GeoDataFrame(network_entsoe.links, geometry = linkGeometry(network_entsoe), crs = "EPSG:4326")

In [ ]:
# Create a GeoDataFrame for the OSM network using the the lineGeometry function
gdf_osm_lines = gpd.GeoDataFrame(network_osm.lines, geometry = lineGeometry(network_osm), crs = "EPSG:4326")
gdf_osm_links = gpd.GeoDataFrame(network_osm.links, geometry = linkGeometry(network_osm), crs = "EPSG:4326")

# Create a GeoDataFrame for the OSM network using accurate LineString geoms
gdf_osm_lines_acc = gpd.GeoDataFrame(network_osm.lines, geometry = network_osm.lines["geometry"], crs = "EPSG:4326")

## Part II: Numerical analysis

### II.1 Preparation

In a next step, we apply different metrics to compare the quality of the two network datasets. To do this on a regional level, we use NUTS2 regions at the highest resolution available, provided in the EPSG:4326 geographic coordinate system (to be consistent with the provided networks). First, we download the latest official data provided by Eurostat and import it using GeoPandas.

In [ ]:
# Set NUTS resolution
nuts = "NUTS1"

if not os.path.exists("output"):
   os.makedirs('output')

In [ ]:
url_nuts2 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_01M_2021_4326_LEVL_2.geojson"
url_nuts1 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_01M_2021_4326_LEVL_1.geojson"
url_nuts2 = "geojson/NUTS_RG_01M_2021_4326_LEVL_2.geojson"
url_nuts1 = "geojson/NUTS_RG_01M_2021_4326_LEVL_1.geojson"

if nuts == "NUTS1":
    gdf_nuts = gpd.read_file(url_nuts1) \
        .drop(columns=["NUTS_ID", "LEVL_CODE", "NUTS_NAME", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE", "FID"])
    output_path = "output/nuts1"
    
elif nuts == "NUTS2":
    gdf_nuts = gpd.read_file(url_nuts2) \
        .drop(columns=["NUTS_ID", "LEVL_CODE", "NUTS_NAME", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE", "FID"])
    output_path = "output/nuts2"

else:
    print("Invalid NUTS resolution. Please choose either 'NUTS1' or 'NUTS2'.")

if not os.path.exists(output_path):
        os.makedirs(output_path)

gdf_nuts.columns

In [ ]:
# Rename columns
gdf_nuts["type"] = nuts
gdf_nuts.rename(columns = {"CNTR_CODE": "country_code", "id": "region_code", "NAME_LATN": "name"}, inplace = True)
gdf_nuts.reindex(columns = ["region_code", "name", "country_code", "type", "geometry"])
gdf_nuts.columns

# Rename Eurostat country code for "EL" to "GR"
gdf_nuts["country_code"] = gdf_nuts["country_code"].replace("EL", "GR")

# Rename region_code prefix "EL" to "GR" for consistency with updated country code for Greece
gdf_nuts["region_code"] = gdf_nuts["region_code"].str.replace("EL", "GR")
gdf_nuts.columns

We see that shapes for BA and UA are missing. To fill the gaps, we use ADM2 geoboundary shapes.

In [ ]:
# BA
url_ba = "https://github.com/wmgeolab/geoBoundaries/raw/905b0ba/" \
    "releaseData/gbOpen/BIH/ADM2/geoBoundaries-BIH-ADM2.geojson"
url_ba="geojson/geoBoundaries-BIH-ADM1.geojson"

gdf_ba = gpd.read_file(url_ba).drop(columns=["shapeID", "shapeGroup", "shapeType"])
gdf_ba.rename(columns = {"shapeName": "name", "shapeISO": "region_code"}, inplace = True)
gdf_ba["region_code"] = gdf_ba["region_code"].str.replace("-", "")
gdf_ba["country_code"], gdf_ba["type"] = ["BA", "ADM1"]
gdf_ba = gdf_ba.reindex(columns = ["region_code", "name", "country_code", "type", "geometry"])
gdf_ba.columns

In [ ]:
# UA
url_ua = "https://github.com/wmgeolab/geoBoundaries/raw/905b0ba/" \
    "releaseData/gbOpen/UKR/ADM1/geoBoundaries-UKR-ADM1.geojson"
url_ua = "geojson/geoBoundaries-UKR-ADM1.geojson"
gdf_ua = gpd.read_file(url_ua).drop(columns=["shapeID", "shapeGroup", "shapeType"])
gdf_ua.rename(columns = {"shapeName": "name", "shapeISO": "region_code"}, inplace = True)
gdf_ua["region_code"] = gdf_ua["region_code"].str.replace("-", "")
gdf_ua["country_code"], gdf_ua["type"] = ["UA", "ADM1"]
gdf_ua = gdf_ua.reindex(columns = ["region_code", "name", "country_code", "type", "geometry"])
gdf_ua.columns

Now let's merge the two GeoDataFrames and remove not needed regions.

In [ ]:
gdf_regions = pd.concat([gdf_nuts, gdf_ba, gdf_ua])
gdf_regions = gdf_regions[(gdf_regions["country_code"] != "IS") & (gdf_regions["country_code"] != "TR")]

# Set index of gdf_regions to region_code
gdf_regions.set_index("region_code", inplace = True)

We create a function that overlays all lines with the provided set of regions. The lines and their geometries (LineStrings) are mapped to the respective region and cut off at the boundary respectively. This is to minimise the amount of distortion/accounting the line length to the wrong region if their are discrepancies between to similar networks.

In [ ]:
# Define function that returns gdf_metrics using the input gdf_regions and 
# gdf_lines
def getMetrics(gdf_regions, gdf_lines):
    print("Obtain list of regions that contain lines.")

    # Obtain a sorted list of region codes that intersect with the lines
    regions_intersect = sorted(
          gpd.overlay(gdf_lines.reset_index(), gdf_regions.reset_index(), how = "intersection")["region_code"] \
            .unique()
          )
    print("Done.\n")
    
    print("Start processing of regions:")
    gdf_metrics = gpd.GeoDataFrame(
        columns=["region_code", "Line", "geometry", "s_nom", "v_nom", "length"], 
        geometry="geometry"
        ) \
          .set_index(["region_code", "Line"])
    
    # for idx_region, region in gdf_regions.iterrows():
    for idx_region in tqdm(regions_intersect, position=0, leave=True):
        gdf_lines_region = gpd.overlay(
            gdf_lines.reset_index(), 
            gdf_regions.loc[[idx_region]].reset_index(), 
            how = "intersection"
            ) \
              .set_index(["region_code", "Line"])
        
        # Calculate length using the estimate_utm_crs method --> returns the length in meters
        utm = gdf_lines_region.loc[[idx_region]].estimate_utm_crs(datum_name = "WGS 84")
        gdf_lines_region.loc[[idx_region], ["length_calc"]] = gdf_lines_region.loc[[idx_region]].to_crs(utm).length/1e3

        gdf_metrics = pd.concat([
            gdf_metrics, 
            gdf_lines_region[["geometry", "s_nom", "v_nom", "length", "length_calc"]]
            ])
        # print("Region", idx_region, "processed.")
    print("Processing finished.\n")

    print("Calculate line volumes...")
    gdf_metrics["line_volume"] = gdf_metrics["length"] * gdf_metrics["s_nom"]
    gdf_metrics["line_volume_calc"] = gdf_metrics["length_calc"] * gdf_metrics["s_nom"]
    print("Complete.")
    
    # Return new GeoDataFrame containing the metrics
    return gdf_metrics

#### Part II.1 - Line volume within region

Now we call the function we have just created on the regions and lines GeoDataFrames.

In [ ]:
# ENTSO-E
gdf_metrics_entsoe = getMetrics(gdf_regions, gdf_entsoe_lines)
gdf_metrics_entsoe.head()

In [ ]:
gdf_metrics_osm = getMetrics(gdf_regions, gdf_osm_lines_acc)
gdf_metrics_osm.head()

Now we sum up the metrics over all regions respectively to obtain the tottal line volumes, whereas 'line_volume" is calculated using the lengths embedded in the PyPSA networks and line_volume_calc is based on the LineStrings.

In [ ]:
# Group metrics entso e by region_code and v_nom:
gdf_metrics_entsoe_grouped = gdf_metrics_entsoe.groupby(["region_code"]) \
    .agg({"length": "sum", "length_calc": "sum", "line_volume": "sum", "line_volume_calc": "sum"})
gdf_metrics_entsoe_grouped.head()

In [ ]:
gdf_metrics_osm_grouped = gdf_metrics_osm.groupby(["region_code"]) \
    .agg({"length": "sum", "length_calc": "sum", "line_volume": "sum", "line_volume_calc": "sum"})
gdf_metrics_osm_grouped.head()

##### Plots

In [ ]:
fig = make_subplots(rows = 1, cols = 2, subplot_titles = ("Line volume", "Calculated line volume"),
shared_yaxes = "all")

fig.add_trace(
    go.Scatter(
        x = gdf_metrics_entsoe_grouped["line_volume"],
        y = gdf_metrics_osm_grouped["line_volume"],
        mode = "markers",
        marker = dict(color = "blue"),
        name = "Line volume (MVAkm)"
        ),
    row = 1, col = 1
    )

fig.add_trace(  
    go.Scatter(
        x = gdf_metrics_entsoe_grouped["line_volume_calc"],
        y = gdf_metrics_osm_grouped["line_volume_calc"],
        mode = "markers",
        marker = dict(color = "green"),
        name = "Calculated line volume (MVAkm)"
        ),
    row = 1, col = 2
    )

# Add labels 
fig.update_traces(
    hovertemplate = "<b>Region</b>: %{text}<br><b>ENTSO-E</b>: %{x} MVAkm<br><b>OSM-E</b>: %{y} MVAkm",
    text = gdf_metrics_osm_grouped.index
    )

fig.update_traces(
    hovertemplate = "<b>Region</b>: %{text}<br><b>ENTSO-E</b>: %{x} MVAkm<br><b>OSM-E</b>: %{y} MVAkm",
    text = gdf_metrics_osm_grouped.index
    )

# Add 1:1 line to both subplots
fig.add_trace(
    go.Scatter(
        x = [0, max(gdf_metrics_entsoe_grouped["line_volume"].max(), 
                    gdf_metrics_osm_grouped["line_volume"].max())],
        y = [0, max(gdf_metrics_entsoe_grouped["line_volume"].max(), 
                    gdf_metrics_osm_grouped["line_volume"].max())],
        mode = "lines",
        line = dict(color = "black", width = 1),
        showlegend = False
        ),
    row = 1, col = 1
    )

fig.add_trace(
    go.Scatter(
        x = [0, max(gdf_metrics_entsoe_grouped["line_volume"].max(), 
                    gdf_metrics_osm_grouped["line_volume"].max())],
        y = [0, max(gdf_metrics_entsoe_grouped["line_volume"].max(), 
                    gdf_metrics_osm_grouped["line_volume"].max())],
        mode = "lines",
        line = dict(color = "black", width = 1),
        showlegend = False
        ),
    row = 1, col = 2
    )

# keep x and y axis ratio equal in all three subplots
fig.update_xaxes(scaleanchor = "x", scaleratio = 1, row = 1, col = 1)
fig.update_xaxes(scaleanchor = "x", scaleratio = 1, row = 1, col = 2)

# make both subplots square 
fig.update_layout(
    title_text = "Comparison of line volumes within " + nuts + " region (MVAkm)",
    width = 1000, height = 500
    )

fig.write_html(output_path + "/scatterplot_line volume.html")
fig.show()

In [ ]:
# Create box plot using two subplots
# Left contains delta between line volumes OSM minus ENTSO-E
# Right contains delta between calculated line volumes OSM minus ENTSO-E

fig = make_subplots(rows = 1, cols = 2, subplot_titles = ("Line volume", "Calculated line volume"),
                    shared_yaxes = "all")

fig.add_trace(
    go.Box(
        y = gdf_metrics_osm_grouped["line_volume"]-gdf_metrics_entsoe_grouped["line_volume"],
        name = "Line volume (MVAkm)",
        marker = dict(color = "blue")
        ),
    row = 1, col = 1
    )

fig.add_trace(
    go.Box(
        y = gdf_metrics_osm_grouped["line_volume_calc"]-gdf_metrics_entsoe_grouped["line_volume_calc"],
        name = "Calculated line volume (MVAkm)",
        marker = dict(color = "green")
        ),
    row = 1, col = 2
    )

fig.update_layout(
    title_text = "Comparison of line volumes deltas (OSM minus ENTSO-E) within " + nuts + " region (MVAkm)",
    width = 1000, height = 500
    )

fig.write_html(output_path + "/boxplot_delta line volume.html")
fig.show()

#### Part II.2 - Transmission capacity between regions

To obtain the transmission capacities between regions, we need to again prepare the data using their geometries and geopandas overlay, intersection and sjoin functions.

In [ ]:
# Create gdf of boundary of regions
gdf_regions_boundary = gdf_regions.copy()
gdf_regions_boundary["geometry"] = gdf_regions_boundary["geometry"].boundary

gdf_intersection_entsoe = gdf_metrics_entsoe.intersection(gdf_regions_boundary)
gdf_intersection_entsoe = gdf_intersection_entsoe[~gdf_intersection_entsoe.is_empty]
gdf_intersects_entsoe = gdf_metrics_entsoe.intersects(gdf_regions_boundary)

gdf_intersection_osm = gdf_metrics_osm.intersection(gdf_regions_boundary)
gdf_intersection_osm = gdf_intersection_osm[~gdf_intersection_osm.is_empty]
gdf_intersects_osm = gdf_metrics_osm.intersects(gdf_regions_boundary)

Note that this methods will also yield lines that intersect more than one time, hence start and end bus of a a line could be in a single region.
This needs to be taken into account and post-processed/removed at a later stage.

In [ ]:
fig_m2 = Figure(width = "50%", height = 600)

m2 = gdf_intersection_entsoe.explore(name = "Lines cross region")
m2 = gdf_regions.explore(m = m2, name = "boundary")

folium.LayerControl(collapsed = False).add_to(m2)

fig_m2.add_child(m2)
m2

Next, we obtain a feeling on how many unique lines are crossing the regions provided.

In [ ]:
list_entsoe_lines_crossregion = gdf_intersection_entsoe[gdf_intersects_entsoe].reset_index()["Line"].unique()
len(list_entsoe_lines_crossregion)

In [ ]:
list_osm_lines_crossregion = gdf_intersection_osm[gdf_intersects_osm].reset_index()["Line"].unique()
len(list_osm_lines_crossregion)

In [ ]:
gdf_entsoe_lines_crossregion = gdf_entsoe_lines[gdf_entsoe_lines.index.isin(list_entsoe_lines_crossregion)]
gdf_osm_lines_crossregion = gdf_osm_lines_acc[gdf_osm_lines_acc.index.isin(list_osm_lines_crossregion)]

In [ ]:
gdf_entsoe_lines_crossregion['start_point'] = gdf_entsoe_lines_crossregion['geometry'] \
    .apply(lambda boundary: boundary.coords[0])
gdf_entsoe_lines_crossregion['end_point'] = gdf_entsoe_lines_crossregion['geometry'] \
    .apply(lambda boundary: boundary.coords[-1])

gdf_osm_lines_crossregion['start_point'] = gdf_osm_lines_crossregion['geometry'] \
    .apply(lambda boundary: boundary.coords[0])
gdf_osm_lines_crossregion['end_point'] = gdf_osm_lines_crossregion['geometry'] \
    .apply(lambda boundary: boundary.coords[-1])

In [ ]:
gdf_entsoe_lines_crossregion_start_points = gpd.GeoDataFrame(
    gdf_entsoe_lines_crossregion[["geometry"]], 
    geometry = gdf_entsoe_lines_crossregion["start_point"].apply(lambda point: Point(point)), 
    crs = "EPSG:4326"
    )

gdf_entsoe_lines_crossregion_start_points = gpd.sjoin(
    gdf_entsoe_lines_crossregion_start_points, 
    gdf_regions.reset_index()[["region_code", "geometry"]], 
    how = "left", 
    op = "within") \
        .drop(columns = ["geometry", "index_right"]
              )

gdf_entsoe_lines_crossregion_end_points = gpd.GeoDataFrame(
    gdf_entsoe_lines_crossregion[["geometry"]], 
    geometry = gdf_entsoe_lines_crossregion["end_point"].apply(lambda point: Point(point)), 
    crs = "EPSG:4326"
    )

gdf_entsoe_lines_crossregion_end_points = gpd.sjoin(
    gdf_entsoe_lines_crossregion_end_points, 
    gdf_regions.reset_index()[["region_code", "geometry"]], 
    how = "left", 
    op = "within") \
        .drop(columns = ["geometry", "index_right"]
              )

gdf_osm_lines_crossregion_start_points = gpd.GeoDataFrame(
    gdf_osm_lines_crossregion[["geometry"]], 
    geometry = gdf_osm_lines_crossregion["start_point"].apply(lambda point: Point(point)),
    crs = "EPSG:4326"
    )

gdf_osm_lines_crossregion_start_points = gpd.sjoin(
    gdf_osm_lines_crossregion_start_points, 
    gdf_regions.reset_index()[["region_code", "geometry"]], 
    how = "left", 
    op = "within") \
        .drop(columns = ["geometry", "index_right"]
              )

gdf_osm_lines_crossregion_end_points = gpd.GeoDataFrame(
    gdf_osm_lines_crossregion[["geometry"]], 
    geometry = gdf_osm_lines_crossregion["end_point"].apply(lambda point: Point(point)), 
    crs = "EPSG:4326"
    )

gdf_osm_lines_crossregion_end_points = gpd.sjoin(
    gdf_osm_lines_crossregion_end_points, 
    gdf_regions.reset_index()[["region_code", "geometry"]], 
    how = "left", 
    op = "within") \
        .drop(columns = ["geometry", "index_right"]
              )

In [ ]:
# Merge gdf_entsoe_lines_crossregion and gdf_entsoe_lines_crossregion_start_
# points by index using left join and rename region_code to region_code_start
gdf_entsoe_lines_crossregion = gdf_entsoe_lines_crossregion.merge(
    gdf_entsoe_lines_crossregion_start_points, 
    left_index = True, 
    right_index = True, how = "left")

gdf_entsoe_lines_crossregion.rename(columns = {"region_code": "region_code_start"}, inplace = True)

# Merge gdf_entsoe_lines_crossregion and gdf_entsoe_lines_crossregion_end_
# points by index using left join and rename region_code to region_code_end
gdf_entsoe_lines_crossregion = gdf_entsoe_lines_crossregion.merge(
    gdf_entsoe_lines_crossregion_end_points, 
    left_index = True, 
    right_index = True, how = "left")

gdf_entsoe_lines_crossregion.rename(columns = {"region_code": "region_code_end"}, inplace = True)

# Drop entries in gdf_entsoe_lines_crossregion that have equal region_code_
# start and region_code_end
gdf_entsoe_lines_crossregion = gdf_entsoe_lines_crossregion[
    gdf_entsoe_lines_crossregion["region_code_start"] != gdf_entsoe_lines_crossregion["region_code_end"]
    ]

gdf_entsoe_lines_crossregion.head()

In [ ]:
# Merge gdf_osm_lines_crossregion and gdf_osm_lines_crossregion_start_points by 
# index using left join and rename region_code to region_code_start
gdf_osm_lines_crossregion = gdf_osm_lines_crossregion.merge(
    gdf_osm_lines_crossregion_start_points, 
    left_index = True, 
    right_index = True, how = "left")

gdf_osm_lines_crossregion.rename(columns = {"region_code": "region_code_start"}, inplace = True)

# Merge gdf_osm_lines_crossregion and gdf_osm_lines_crossregion_end_points by 
# index using left join and rename region_code to region_code_end
gdf_osm_lines_crossregion = gdf_osm_lines_crossregion.merge(
    gdf_osm_lines_crossregion_end_points, 
    left_index = True, 
    right_index = True, how = "left")

gdf_osm_lines_crossregion.rename(columns = {"region_code": "region_code_end"}, inplace = True)

# Drop entries in gdf_osm_lines_crossregion that have equal region_code_start 
# and region_code_end
gdf_osm_lines_crossregion = gdf_osm_lines_crossregion[gdf_osm_lines_crossregion["region_code_start"] != gdf_osm_lines_crossregion["region_code_end"]]

gdf_osm_lines_crossregion.head()

In [ ]:
gdf_entsoe_lines_crossregion_grouped = gdf_entsoe_lines_crossregion.groupby(["region_code_start", "region_code_end"]) \
    .agg({"s_nom": [("s_nom_sum", "sum"), ("s_nom_count", "count")]})
gdf_entsoe_lines_crossregion_grouped.columns = gdf_entsoe_lines_crossregion_grouped.columns.droplevel(0)

gdf_osm_lines_crossregion_grouped = gdf_osm_lines_crossregion.groupby(["region_code_start", "region_code_end"]) \
    .agg({"s_nom": [("s_nom_sum", "sum"), ("s_nom_count", "count")]})
gdf_osm_lines_crossregion_grouped.columns = gdf_osm_lines_crossregion_grouped.columns.droplevel(0)

In [ ]:
# Add new column "region_code_start_sorted" that takes either the value from 
# region_code_start or region_code_end depending on the lexicographical order 
# of the two values
gdf_entsoe_lines_crossregion_grouped["region_code_start_sorted"] = gdf_entsoe_lines_crossregion_grouped \
    .apply(lambda row: row.name[0] if row.name[0] < row.name[1] else row.name[1], axis = 1)
gdf_entsoe_lines_crossregion_grouped["region_code_end_sorted"] = gdf_entsoe_lines_crossregion_grouped \
    .apply(lambda row: row.name[1] if row.name[0] < row.name[1] else row.name[0], axis = 1)

gdf_entsoe_lines_crossregion_grouped.reset_index().groupby(["region_code_start_sorted", "region_code_end_sorted"]) \
    .agg({"s_nom_sum": "sum", "s_nom_count": "sum"})

In [ ]:
# Add new column "region_code_start_sorted" that takes either the value from 
# region_code_start or region_code_end depending on the lexicographical order 
# of the two values
gdf_osm_lines_crossregion_grouped["region_code_start_sorted"] = gdf_osm_lines_crossregion_grouped \
    .apply(lambda row: row.name[0] if row.name[0] < row.name[1] else row.name[1], axis = 1)
gdf_osm_lines_crossregion_grouped["region_code_end_sorted"] = gdf_osm_lines_crossregion_grouped \
    .apply(lambda row: row.name[1] if row.name[0] < row.name[1] else row.name[0], axis = 1)

gdf_osm_lines_crossregion_grouped.reset_index().groupby(["region_code_start_sorted", "region_code_end_sorted"]) \
    .agg({"s_nom_sum": "sum", "s_nom_count": "sum"})

In [ ]:
# merge gdf_osm_lines_crossregion_grouped and gdf_entsoe_lines_crossregion_
# grouped by region_code_start_sorted and region_code_end_sorted using outer 
# join  
gdf_lines_crossregion = gdf_entsoe_lines_crossregion_grouped.merge(
    gdf_osm_lines_crossregion_grouped, 
    left_on = ["region_code_start_sorted", "region_code_end_sorted"], 
    right_on = ["region_code_start_sorted", "region_code_end_sorted"], 
    suffixes = ["_entsoe", "_osm"], 
    how = "outer"
    )

In [ ]:
gdf_lines_crossregion_grouped = gdf_lines_crossregion.groupby(["region_code_start_sorted", "region_code_end_sorted"]) \
    .agg({"s_nom_sum_entsoe": "sum", "s_nom_count_entsoe": "sum", "s_nom_sum_osm": "sum", "s_nom_count_osm": "sum"}, 
         min_count = 1)
gdf_lines_crossregion_grouped

In [ ]:
# Find zero outliers in both datasets
gdf_lines_crossregion_grouped[(gdf_lines_crossregion_grouped["s_nom_count_osm"] == 0) | 
                              (gdf_lines_crossregion_grouped["s_nom_count_entsoe"] == 0)]

In [ ]:
fig = make_subplots(rows = 1, cols = 2, subplot_titles = ("S_nom (MVA)", "No. of lines crossing region"))

fig.add_trace(
    go.Scatter(
        x = gdf_lines_crossregion_grouped["s_nom_sum_entsoe"],
        y = gdf_lines_crossregion_grouped["s_nom_sum_osm"],
        mode = "markers",
        marker = dict(color = "blue"),
        name = "S_nom (MVA)"
        ),
    row = 1, col = 1
    )

fig.add_trace(
    go.Scatter(
        x = gdf_lines_crossregion_grouped["s_nom_count_entsoe"],
        y = gdf_lines_crossregion_grouped["s_nom_count_osm"],
        mode = "markers",
        marker = dict(color = "green"),
        name = "No. of lines"
        ),
    row = 1, col = 2
    )


# Add labels 
fig.update_traces(
    hovertemplate = "<b>Region</b>: %{text}<br><b>ENTSO-E</b>: %{x:.0f} MVA<br><b>OSM-E</b>: %{y:.0f} MVA",
    text = gdf_lines_crossregion_grouped.index,
    row = 1, col = 1
    )

fig.update_traces(
    hovertemplate = "<b>Region</b>: %{text}<br><b>ENTSO-E</b>: %{x} lines<br><b>OSM-E</b>: %{y} lines",
    text = gdf_lines_crossregion_grouped.index,
    row = 1, col = 2
    )

# Add 1:1 line to both subplots
fig.add_trace(
    go.Scatter(
        x = [0, max(gdf_lines_crossregion_grouped["s_nom_sum_entsoe"].max(), 
                    gdf_lines_crossregion_grouped["s_nom_sum_osm"].max())],
        y = [0, max(gdf_lines_crossregion_grouped["s_nom_sum_entsoe"].max(), 
                    gdf_lines_crossregion_grouped["s_nom_sum_osm"].max())],
        mode = "lines",
        line = dict(color = "black", width = 1),
        showlegend = False
        ),
    row = 1, col = 1
    )

fig.add_trace(
    go.Scatter(
        x = [0, max(gdf_lines_crossregion_grouped["s_nom_count_entsoe"].max(), 
                    gdf_lines_crossregion_grouped["s_nom_count_osm"].max())],
        y = [0, max(gdf_lines_crossregion_grouped["s_nom_count_entsoe"].max(), 
                    gdf_lines_crossregion_grouped["s_nom_count_osm"].max())],
        mode = "lines",
        line = dict(color = "black", width = 1),
        showlegend = False
        ),
    row = 1, col = 2
    )

# keep x and y axis ratio equal in all three subplots
fig.update_xaxes(scaleanchor = "x", scaleratio = 1, row = 1, col = 1)
fig.update_xaxes(scaleanchor = "x", scaleratio = 1, row = 2, col = 1)

# make both subplots square 
fig.update_layout(
    title_text = "Cross-region capacity (MVA) between " + nuts + " regions",
    width = 1000, height = 500
    )

fig.write_html(output_path + "/scatterplot_cross-region capacity and no of lines.html")

fig.show()

## Part III: Visualisation

In [ ]:
# Create colormap using unique voltage levels from ENTSO-E and OSM data
list_v_nom = np.concatenate(
    [gdf_entsoe_lines.v_nom.unique(), 
    gdf_osm_lines.v_nom.unique()]
    )

list_v_nom = np.unique(list_v_nom)
list_v_nom

colormap = cm.LinearColormap(
    ["orange", "purple", "black"], 
    vmin = gdf_entsoe_lines["v_nom"].min(),
    vmax = gdf_entsoe_lines["v_nom"].max()
    )

colormap = colormap.to_step(index = list_v_nom)
colormap.caption = "Voltage level in kV"
colormap

In [ ]:
# Substations
buses_xy_entsoe = gpd.points_from_xy(network_entsoe.buses.x, network_entsoe.buses.y)
buses_xy_osm = gpd.points_from_xy(network_osm.buses.x, network_osm.buses.y)

gdf_entsoe_buses = gpd.GeoDataFrame(geometry = buses_xy_entsoe, crs = "EPSG:4326")
gdf_osm_buses = gpd.GeoDataFrame(geometry = buses_xy_osm, crs="EPSG:4326")

In [ ]:
gdf_regions

In [ ]:
# Define tooltip that is shown when hovering over the lines
def tooltip_lines():
    tooltip = folium.GeoJsonTooltip(
        fields=["Line", "bus0", "bus1", "s_nom", "v_nom", "length"],
        aliases=["Line:", "Bus 0:", "Bus 1:", "Capacity (MVA):", "Voltage (kV):", "Length (km):"],
        labels = True,
        localize = True,
        sticky = False
    )
    return tooltip

def tooltip_links():
    tooltip = folium.GeoJsonTooltip(
        fields=["Link", "bus0", "bus1", "p_nom", "length"],
        aliases=["Line:", "Bus 0:", "Bus 1:", "P (nom.) (MW):", "Length (km):"],
        labels = True,
        localize = True,
        sticky = False
    )
    return tooltip

def tooltip_regions():
    tooltip = folium.GeoJsonTooltip(
        fields=["region_code", "name", "type"],
        aliases=["Region code:", "Region name:", "Type:"],
        labels = True,
        localize = True,
        sticky = False,
    )
    return tooltip

# Determine the center of the map
loc_x = np.array([network_entsoe.buses.x.min(), network_entsoe.buses.x.max()]).mean()
loc_y = np.array([network_entsoe.buses.y.min(), network_entsoe.buses.y.max()]).mean()

# Create the folium map using CartoDB positron tiles
m = folium.plugins.DualMap(location=(loc_y, loc_x), zoom_start=7, tiles = "CartoDB positron")

m1_title = "ENTSO-E network"
m2_title = "OSM network"

m.m1.get_root().html.add_child(
    folium.Element(f"<h1 style='position:absolute;z-index:100000;left:20vw'>{m1_title}</h1>")
    )

m.m2.get_root().html.add_child(
    folium.Element(f"<h1 style='position:absolute;z-index:100000;right:20vw'>{m2_title}</h1>")
    )

# Add the colormap (legend) 
colormap.add_to(m.m2)

# Add the feature groups to the map
fg_entsoe_buses = folium.FeatureGroup(name = "Buses ENTSO-E").add_to(m.m1)
fg_osm_buses = folium.FeatureGroup(name = "Buses OSM").add_to(m.m2)

fg_entsoe_lines = folium.FeatureGroup(name = "Lines ENTSO-E").add_to(m.m1)
fg_osm_lines = folium.FeatureGroup(name = "Lines OSM", show = False).add_to(m.m2)
fg_osm_lines_acc = folium.FeatureGroup(name = "Lines OSM (acc.)").add_to(m.m2)

fg_entsoe_links = folium.FeatureGroup(name = "Links ENTSO-E").add_to(m.m1)
fg_osm_links = folium.FeatureGroup(name = "Links OSM").add_to(m.m2)

fg_entsoe_crossreg = folium.FeatureGroup(name = nuts + ": Cross-region ENTSO-E ", show = False).add_to(m.m1)
fg_osm_crossreg = folium.FeatureGroup(name = nuts + ": Cross-region OSM", show = False).add_to(m.m2)

# Define the callback function for the FastMarkerCluster
callback = (
    'function (row) {'
    'var circle = L.circle(new L.LatLng(row[0], row[1]), {color: "red",  radius: 1500});'
    'return circle};'
    )

# Add the layer control to the map
folium.LayerControl(collapsed = False).add_to(m)

# Add the buses to the feature groups
folium.plugins.FastMarkerCluster(
    network_entsoe.buses[["y", "x"]].values.tolist(), 
    callback = callback, disableClusteringAtZoom = 5,
    ) \
        .add_to(fg_entsoe_buses)

folium.plugins.FastMarkerCluster(
    network_osm.buses[["y", "x"]].values.tolist(), 
    callback = callback, disableClusteringAtZoom = 5,
    fill_opacity = 0.1
    ) \
        .add_to(fg_osm_buses)

# Add the lines to the feature groups
folium.GeoJson(
    gdf_entsoe_lines.reset_index(), 
    style_function = lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_entsoe_lines)

folium.GeoJson(
    gdf_osm_lines.reset_index(), 
    style_function = lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_osm_lines)

folium.GeoJson(
    gdf_osm_lines_acc.reset_index(), 
    style_function = lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_osm_lines_acc)

# Add the links to the feature groups
folium.GeoJson(
    gdf_entsoe_links.reset_index(), 
    style_function = lambda feature: {
        # "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_links(),
    ) \
        .add_to(fg_entsoe_links)

folium.GeoJson(
    gdf_osm_links.reset_index(), 
    style_function = lambda feature: {
        # "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_links(),
    ) \
        .add_to(fg_osm_links)

### Cross-region visualisations
folium.GeoJson(
    gdf_regions.reset_index(), 
    style_function = lambda feature: {
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_regions(),
    ) \
        .add_to(fg_entsoe_crossreg)

folium.GeoJson(
    gdf_regions.reset_index(), 
    style_function = lambda feature: {
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_regions(),
    ) \
        .add_to(fg_osm_crossreg)

folium.GeoJson(
    gdf_entsoe_lines_crossregion.reset_index(), 
    style_function = lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_entsoe_crossreg)

folium.GeoJson(
    gdf_osm_lines_crossregion.reset_index(), 
    style_function = lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_osm_crossreg)

# Export the map to a self-contained .html file
m.save(output_path + "/map.html")

Lets explore the two networks side-by-side using leaflet/folium below.

In [ ]:
# Display interactive map
m

In [ ]:
# fig_m3 = Figure(width = "50%", height = 600)

# m3 = gdf_entsoe_lines_crossregion.explore(name = "Lines cross region", color = "orange")
# m3 = gdf_regions.explore(m = m3, name = "boundary")
# m3 = gpd.GeoDataFrame(geometry = gdf_entsoe_lines_crossregion["start_point"] \
#   .apply(lambda point: Point(point)), crs = "EPSG:4326").explore(m = m3, name = "start_point", color = "red")
# m3 = gpd.GeoDataFrame(geometry = gdf_entsoe_lines_crossregion["end_point"] \
#   .apply(lambda point: Point(point)), crs = "EPSG:4326").explore(m = m3, name = "end_point", color = "green")

# folium.LayerControl(collapsed = False).add_to(m3)

# fig_m3.add_child(m3)
# m3